In [2]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import datetime as dt

In [3]:
# Parameters
min_rest_days = 3
max_matches_per_venue_day = 1
max_matches_per_kickoff = 20

# Weights for fan heat exposure indoor and outdoor
# 80% time spent indoors / 20% time spent outdoors
w_in = 0.8
w_out = 0.2

# FIFA matchday windows
matchday_dates = {
    1: [
        dt.date(2026, 6, 11),
        dt.date(2026, 6, 12),
        dt.date(2026, 6, 13),
        dt.date(2026, 6, 14),
        dt.date(2026, 6, 15),
        dt.date(2026, 6, 16)
    ],
    2: [
        dt.date(2026, 6, 17),
        dt.date(2026, 6, 18),
        dt.date(2026, 6, 19),
        dt.date(2026, 6, 20),
        dt.date(2026, 6, 21),
        dt.date(2026, 6, 22)
    ],
    3: [
        dt.date(2026, 6, 23),
        dt.date(2026, 6, 24),
        dt.date(2026, 6, 25),
        dt.date(2026, 6, 26),
        dt.date(2026, 6, 27)
    ]
}

matches = pd.read_csv("output.csv")
wbgt = pd.read_csv("WBGT_data.csv")

# Convert CSV dates from strings to Python date objects
wbgt["Date"] = pd.to_datetime(wbgt["Date"]).dt.date

In [5]:
# Map kickoff times to CSV columns
kickoff_columns = {
    "12:00": "WBGT_12",
    "15:00": "WBGT_15",
    "18:00": "WBGT_18",
    "21:00": "WBGT_21"
}

# All possible schedule options
options = []
option_id = 0

for _, match in matches.iterrows():

    match_id = match["match_id"]
    matchday = int(match["matchday"])

    team1 = match["team_1"]
    team2 = match["team_2"]

    stadium = match["stadium"]
    capacity = match["capacity"]

    # Only use dates belonging to this matchday
    for date in matchday_dates[matchday]:

        # Find WBGT data for this stadium/city and date
        city_weather = wbgt[
            (wbgt["City"] == stadium) &
            (wbgt["Date"] == date)
        ]

        if city_weather.empty:
            print("No data for: ", stadium, date)

        weather = city_weather.iloc[0]

        # Try each possible kickoff time
        for kickoff_time, wbgt_column in kickoff_columns.items():

            outdoor_wbgt = weather[wbgt_column]

            # Indoor WBGT
            if weather["Climate_Controlled"] == 1:
                indoor_wbgt = 21.0
            else:
                indoor_wbgt = outdoor_wbgt

            # Weighted heat risk
            risk = (
                w_in * indoor_wbgt +
                w_out * outdoor_wbgt
            )

            options.append({
                "option_id": option_id,

                "match_id": match_id,
                "group": match["group"],
                "team1": team1,
                "team2": team2,

                "matchday": matchday,
                "stadium": stadium,
                "capacity": capacity,

                "date": date,
                "date_idx": (date - dt.date(2026, 1, 1)).days,
                "kickoff_time": kickoff_time,

                "wbgt_outdoor": outdoor_wbgt,
                "wbgt_indoor": indoor_wbgt,

                "risk": round(risk, 2)
            })

            option_id += 1

# Combine all schedule options
schedule_options = (
    pd.DataFrame(options)
    .set_index("option_id")
)

print(f"Total schedule options = {len(schedule_options)}")

schedule_options.head()

Total schedule options = 1632


,match_id,group,team1,team2,matchday,stadium,capacity,date,date_idx,kickoff_time,wbgt_outdoor,wbgt_indoor,risk
option_id,,,,,,,,,,,,,
0,A_1,A,Argentina,Mexico,1,Monterrey,51243,2026-06-11,161,12:00,28.41,28.41,28.41
1,A_1,A,Argentina,Mexico,1,Monterrey,51243,2026-06-11,161,15:00,28.17,28.17,28.17
2,A_1,A,Argentina,Mexico,1,Monterrey,51243,2026-06-11,161,18:00,27.17,27.17,27.17
3,A_1,A,Argentina,Mexico,1,Monterrey,51243,2026-06-11,161,21:00,25.36,25.36,25.36
4,A_1,A,Argentina,Mexico,1,Monterrey,51243,2026-06-12,162,12:00,28.80,28.80,28.80


In [6]:
m = gp.Model("wbgt_heat_risk_scheduling")

# Decision variables
x = {}

# Store available options for each match
options_for_match = {}

for match_id in matches["match_id"]:

    valid_options = schedule_options[
        schedule_options["match_id"] == match_id
    ].index.tolist()

    options_for_match[match_id] = valid_options

    for option_id in valid_options:

        x[match_id, option_id] = m.addVar(vtype=GRB.BINARY)

m.update()

print(f"Decision variables: {len(x)}")

Set parameter Username
Set parameter LicenseID to value 2826745
Academic license - for non-commercial use only - expires 2027-05-22
Decision variables: 1632


In [7]:
# Minimize heat-risk exposure based on stadium capacity and WBGT risk for each schedule option
obj = gp.quicksum(
    schedule_options.loc[option_id, "risk"]
    * schedule_options.loc[option_id, "capacity"]
    * x[match_id, option_id]

    for match_id, options in options_for_match.items()
    for option_id in options
)

m.setObjective(obj, GRB.MINIMIZE)

In [8]:
# Every match scheduled exactly once
for match_id, options in options_for_match.items():

    m.addConstr(
        gp.quicksum(
            x[match_id, option_id]
            for option_id in options
        ) == 1
    )

# Maximum of 1 match at each stadium per day
for (stadium, date), option_ids in schedule_options.groupby(
    ["stadium", "date"]
).groups.items():

    match_options = [
        x[match_id, option_id]

        for match_id, options in options_for_match.items()
        for option_id in options

        if option_id in option_ids
    ]

    if match_options:

        m.addConstr(
            gp.quicksum(match_options)
            <= max_matches_per_venue_day
        )

# Get the selected date for a match
def get_match_date(match_id):

    date = gp.LinExpr()

    for option_id in options_for_match[match_id]:

        date += (
            schedule_options.loc[
                option_id,
                "date_idx"
            ]
            * x[match_id, option_id]
        )

    return date

# Teams must have at least 3 rest days between matches
for group, group_matches in matches.groupby("group"):

    teams = set(
        group_matches["team_1"]
    ).union(
        group_matches["team_2"]
    )

    for team in teams:

        team_matches = []

        # Find this team's match in each round
        for matchday in [1, 2, 3]:

            matchday_matches = group_matches[
                group_matches["matchday"] == matchday
            ]

            team_match = matchday_matches[
                (matchday_matches["team_1"] == team) |
                (matchday_matches["team_2"] == team)
            ]

            if len(team_match) == 1:

                team_matches.append(
                    team_match.iloc[0]["match_id"]
                )

        # Ensure rest between consecutive matches
        for previous_match, next_match in zip(
            team_matches,
            team_matches[1:]
        ):

            m.addConstr(
                get_match_date(next_match)
                - get_match_date(previous_match)
                >= min_rest_days
            )

# Maximum number of matches at each kickoff time
for kickoff in schedule_options["kickoff_time"].unique():

    kickoff_options = [
        x[match_id, option_id]

        for match_id, options in options_for_match.items()
        for option_id in options

        if schedule_options.loc[
            option_id,
            "kickoff_time"
        ] == kickoff
    ]

    m.addConstr(
        gp.quicksum(kickoff_options)
        <= max_matches_per_kickoff
    )

m.update()

print(f"Total constraints: {m.NumConstrs}")

Total constraints: 444


In [9]:
m.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i5-1335U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 444 rows, 1632 columns and 9312 nonzeros (Min)
Model fingerprint: 0xba12eb88
Model has 1632 linear objective coefficients
Variable types: 0 continuous, 1632 integer (1632 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [6e+05, 2e+06]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]

Found heuristic solution: objective 9.996195e+07
Presolve removed 162 rows and 409 columns
Presolve time: 0.04s
Presolved: 282 rows, 1223 columns, 5257 nonzeros
Found heuristic solution: objective 9.620297e+07
Variable types: 0 continuous, 1223 integer (1223 binary)

Root relaxation: objective 8.523928e+07, 225 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |  

In [10]:
results = []

if m.SolCount > 0:

    for match_id, options in options_for_match.items():

        for option_id in options:

            if x[match_id, option_id].X > 0.5:

                option = schedule_options.loc[option_id]

                results.append({
                    "group": option["group"],
                    "match_id": option["match_id"],
                    "team1": option["team1"],
                    "team2": option["team2"],
                    "matchday": option["matchday"],
                    "stadium": option["stadium"],
                    "capacity": option["capacity"],

                    "date": option["date"],
                    "kickoff_time": option["kickoff_time"],

                    "wbgt_outdoor": option["wbgt_outdoor"],
                    "wbgt_indoor": option["wbgt_indoor"],
                    "risk": option["risk"],

                    "person_weighted_risk":
                        option["risk"]
                        * option["capacity"]
                })

    results_df = (
        pd.DataFrame(results)
        .sort_values(
            ["group", "matchday"]
        )
        .reset_index(drop=True)
    )

    print("Total person-weighted risk:", m.ObjVal)
    print("Average WBGT per match:", results_df["risk"].mean())
    results_df

else:

    print("No feasible solution found.")

Total person-weighted risk: 85656313.01000002
Average WBGT per match: 18.739444444444445


In [30]:
results_df.to_csv(
    "optimized_heat_schedule.csv",
    index=False
)

Below, we compare the official FIFA schedule's results against our current model for calculating heat-risk.

In [12]:
# Load official FIFA schedule
real_schedule = pd.read_csv("real_schedule.csv")

# Convert dates
real_schedule["date"] = pd.to_datetime(
    real_schedule["date"]
).dt.date

# Round kickoff times to match available WBGT data
def get_wbgt_column(kickoff_time):
    hour = int(kickoff_time.split(":")[0])

    if hour <= 13:
        return "WBGT_12"
    elif hour <= 16:
        return "WBGT_15"
    elif hour <= 19:
        return "WBGT_18"
    else:
        return "WBGT_21"

risks = []
outdoor_wbgt_values = []
indoor_wbgt_values = []

for _, match in real_schedule.iterrows():

    weather = wbgt[
        (wbgt["City"] == match["city"]) &
        (wbgt["Date"] == match["date"])
    ].iloc[0]

    # Match actual kickoff to closest available WBGT time
    wbgt_column = get_wbgt_column(
        match["kickoff_time_local"]
    )

    outdoor_wbgt = weather[wbgt_column]

    if weather["Climate_Controlled"] == 1:
        indoor_wbgt = 21.0
    else:
        indoor_wbgt = outdoor_wbgt

    risk = (
        w_in * indoor_wbgt +
        w_out * outdoor_wbgt
    )

    outdoor_wbgt_values.append(outdoor_wbgt)
    indoor_wbgt_values.append(indoor_wbgt)
    risks.append(risk)


real_schedule["risk"] = risks

real_schedule["person_weighted_risk"] = (
    real_schedule["risk"] *
    real_schedule["capacity"]
)

print("Total person-weighted risk:", real_schedule["person_weighted_risk"].sum())
print("Average WBGT per match:", real_schedule["risk"].mean())

Total person-weighted risk: 97400857.07000001
Average WBGT per match: 20.809916666666666
